In [1]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.8/262.8 KB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 KB 68.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 KB 42.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 148.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 4.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 35.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 167.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 187.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 KB 69.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 KB 65.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.2/29.2 MB 74.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 60.9 MB/s eta 0:00:0000

In [30]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered.en.subword.train
        path_tgt: en-zh.zh-filtered.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered.en.subword.dev
        path_tgt: en-zh.zh-filtered.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.base

# Stop training if it does not imporve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [3]:
# Find the number of CPUs/cores on the machine
!nproc --all

16


In [28]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 16


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_build_vocab", line 5, in <module>
    from onmt.bin.build_vocab import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "

In [5]:
# Check if the GPU is active
!nvidia-smi -L

GPU 0: NVIDIA L40S (UUID: GPU-91006845-7b0b-d094-209b-a3a40a8ccfc6)


In [6]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/venv/main/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/venv/main/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/venv/main/lib/python3.10/site-packages/ipykernel/kernelapp.p

True
NVIDIA L40S
Free GPU memory: 45158.25 out of: 45589.0625


In [31]:
# Train the NMT model
!onmt_train -config config.yaml


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_train", line 5, in <module>
    from onmt.bin.train import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/venv/main/l

## Translate

In [36]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
# gpu
!onmt_translate -model models/model.base_step_10000.pt -src en-zh.en-filtered.en.subword.test -output zh.base.translated -gpu 0 -min_length 1




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/venv/main/bin/onmt_translate", line 5, in <module>
    from onmt.bin.translate import main
  File "/venv/main/lib/python3.10/site-packages/onmt/__init__.py", line 2, in <module>
    import onmt.inputters
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/__init__.py", line 7, in <module>
    from onmt.inputters.text_utils import text_sort_key, process, numericalize, tensorify
  File "/venv/main/lib/python3.10/site-packages/onmt/inputters/text_utils.py", line 1, in <module>
    import torch
  File "/ven

In [14]:
%pip install "numpy<2"


Note: you may need to restart the kernel to use updated packages.


In [20]:
# Check the first 5 lines of the translation file
!head -n 30 zh.translated

▁在 捷 克斯 洛 伐 克 , 东 德国 , ▁ 爱 沙 尼亚 、 拉 特 维 亚 、 马 里 兰 、 马 里 亚 、 马 达 加 斯 加 、 ▁ 菲 律 宾 、 塞 罗 维 亚 、 斯 洛 文 尼亚 、 现在 和 埃及 。
▁我不知道 他们 要 怎么 处理 这些东西 。
▁它 需要 一些 审 美 的概念 , ▁因为它 是一个 没有 雕 刻 的 系统 。
▁ 伟大的 老师 这么做 了 ▁但是 伟大的 老师 也在 做的 ▁是 辅 导 , 激励 着 , 激励 着 , 激励 着 , 激励 着
▁所以我 只是 插 上 插 入 这个 树 枝 型 的工作 , ▁最后 , 在 4 0 0 0 个 尝试 中 , ▁当我 接近 失去 理智 的时候 , 我 发现了 蛋白质 。
▁我 注意到 他们 将 通过 1 度 或 2 度 的 温度 ▁来 改变 家里 的 温度 。
▁但是 这是 结果 。
▁所以 “ 第二 人生 ” 里 一个人 的 平均 年龄 是 3 2 岁 , ▁但是 “ 第二 人生 ” 的 使用 ▁ 也会 随着 你的 实际 年龄 的 增长 而 急 剧 增加 。 ▁所以 , 当你 从 3 0 岁 到 6 0 岁 , ▁他们 中的 6 0 0 个 人口 ▁ 使用 “ 第二 人生 ” , ▁ 这并不是 一个 明显 的 曲线 , ▁非常 分散 的 , 就像 每 星期 4 0 % 的 使用 量 。
▁它 不停地 想 它 自己的 抽象 图案 。
▁ 这项技术 的 好处 在于 ▁ 这项技术 能 使 手机 开始 看到 ▁并 了解 人类 大脑的 运作 方式 。
▁ 建筑师 的 帮助 下 ▁ 当地 居民 毫不 夸张 的 从 地 上 养 活 自己
▁然后 学生们 会 加入 我们 的声音 工作室 ▁他们会 用 他们自己的 韵 律 ▁ 创作 出 唱 歌曲
▁有时 我 从 7 日 的 阿 维 特 斯 · 教堂 ▁ 给我 看 这些 关于 天堂 的 卡通 画 。
▁如果你 把 杯子 和 ▁ 小 农场 主 的 农业 填 补 , ▁你 将会 得到 改变 。
▁ 过了 一 小时 , 她 找到 他 , 说 ,“ 你 是谁 ?”
▁现在 , 非常 明显 地 , 我将 这 副 牌 聚集 到一起 。
▁这是一个 完整的 集 成 系统 , ▁尽管 有 规划 等等 。
▁然后 现代 人 出现 在非洲 某 处 ▁ 估计 在 中东 

In [37]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.base.translated

Done desubwording! Output: zh.base.translated.desubword


In [38]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered.en.subword.test

# Desubword the test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.zh-filtered.zh.subword.test

Done desubwording! Output: en-zh.en-filtered.en.subword.test.desubword
Done desubwording! Output: en-zh.zh-filtered.zh.subword.test.desubword


In [39]:
# Check the first 5 lines of the desubworded translation file
!head -n 30 zh.base.translated.desubword

在捷克洛伐克的达基地, 东德国,爱沙尼亚、拉塔维亚、马斯达加、马里斯加、马达加斯加、 宾夕法尼亚、塞尔维亚、斯洛文尼亚、 斯洛文尼亚、突尼西亚、埃及。
我不知道他们要怎么处理那些东西。
它要求某种看证的概念 是一个没有雕刻的系统。
伟大的教师这样做,但是伟大的教师们也是这样, 指导、激励和投入。
所以我只是在方圆柱的任务上加力减速, 最终,在4000次努力中, 当我靠近我的正常线时, 我发现蛋白质。
我注意到,它们会 通过一度或两度的温度来改变家庭的温度。
这是个后果。
所以“第二人生”的平均年龄是32岁, 然而“第二人生”的使用率 变得越来越大。所以当你从30岁到60岁-- 有很多人在使用“第二人生”中使用“第二人生”-- 这不是一个很尖锐的曲线-- 它的分布情况非常显著。
所以它不停地想清楚它自己的抽象。
这项技术的好处在于 科技实际上使手机 开始认识和理解 人脑如何运作
建筑师的帮助下,将居民们从地上抬起来。
然后学生会加入我们的声音室, 他们会用他们自己的节奏创造他们自己的说唱诗。
有时我从第七天的老太太太太太太太 小心 教会了我一些天堂的照片。
如果你用当地农业来填满 小农场主的水源, 你就有了变化的效果。
然后,她在他一小时后到达, 他说,“你是谁?”
现在,非常明显的,我要把这些牌聚集起来。
它是一个综合的系统,尽管有规划,等等。
然后现代人类出现在非洲的某个地方, 很可能来自于中东。
这些干扰对我们解决问题, 帮助我们变得更有创造力。
格伦看起来有点老了。
如果可以的话,那不是“棒极了”吗? 第一次看到这样的眼皮内 能完全适应你, 而且不需要任何的打猎物, 所以更有可能的是,  ⁇ 虫不会摔碎吗?
现在,中国的经济增长 是严峻的改变,基础上的改变, 25年前,发展中国家, 穷国,尽管是, 依然是大多数人的, 他们只占了世界总产量的三分之一。
当然了,现在在家中我非常敏感 当我们把灯熄灭时。
表现最小的国家,中共和国, 31分。
这些都是天然的人类冲动, 但是因为技术, 执行这些冲动只是一次点击。
这是我第一次观察到的, 当我在西班牙海湾上第一次跳下水时。
这本身也存在于世界杯赛选中。
我们确实找到了。
我知道有消防员的人告诉我 这一点也不稀奇。
不,我认为他谈的是选择性的森林。


## Evaluation

In [24]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-04 05:38:33--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
200 OKequest sent, awaiting response... 
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py.2’

compute-bleu.py.2   100%[===================>]     957  --.-KB/s    in 0s      

2025-04-04 05:38:33 (44.1 MB/s) - ‘compute-bleu.py.2’ saved [957/957]



In [25]:
# Install sacrebleu
!pip3 install sacrebleu

In [40]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered.zh.subword.test.desubword zh.base.translated.desubword

Reference 1st sentence: 在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔维亚,斯洛维尼亚的独裁政府,我可以继续, 还有现在的突尼斯和埃及。
MTed 1st sentence: 在捷克洛伐克的达基地, 东德国,爱沙尼亚、拉塔维亚、马斯达加、马里斯加、马达加斯加、 宾夕法尼亚、塞尔维亚、斯洛文尼亚、 斯洛文尼亚、突尼西亚、埃及。
BLEU:  3.1733283687218847


In [ ]:
# !python3 compute-meteor.py en-zh.zh-filtered.zh.subword.test.desubword zh.base.translated.desubword

In [ ]:
# !python3 compute-comet.py en-zh.zh-filtered.zh.subword.test.desubword zh.base.translated.desubword

In [ ]:
# !python3 compute-chrf.py en-zh.zh-filtered.zh.subword.test.desubword zh.base.translated.desubword

In [ ]:
# !git clone https://github.com/google-research/bleurt.git | pip install bleurt/. # note this will create a new folder in the current directory which will be bulky, change the path if needed
# !python3 compute-bleurt.py en-zh.zh-filtered.zh.subword.test.desubword zh.base.translated.desubword

Base Model (fine-tuned):

BLEU:  3.1733283687218847

METEOR (zh): 0.3710 [0, 1]

chrF: 20.3014 [0, 100]

BLEURT: 0.4522 [0, 1]

COMET System Score: 0.7151 [0, 1]

Salient Model:

BLEU:  3.37646848432582 [0, 100]

METEOR (zh): 0.3691 [0, 1]

chrF: 20.3759 [0, 100]

BLEURT: 0.4384 [0, 1]

COMET System Score: 0.7162 [0, 1]


WSD Model:

BLEU:  2.3900016239366977 [0, 100]

METEOR (zh): 0.3074 [0, 1]

chrF: 15.4311 [0, 100]

BLEURT: 0.4083 [0, 1]

COMET System Score: 0.6367 [0, 1]

In [ ]:
# df

,BLEU,METEOR (zh),chrF,BLEURT,COMET System Score
Model,,,,,
Base Model (fine-tuned),3.173328,0.3710,20.3014,0.4522,0.7151
Salient Model,3.376468,0.3691,20.3759,0.4384,0.7162
WSD Model,2.390002,0.3074,15.4311,0.4083,0.6367
